In [3]:
import sys
!{sys.executable} -m pip install faker pyodbc

  Using cached faker-40.21.0-py3-none-any.whl.metadata (16 kB)
Using cached faker-40.21.0-py3-none-any.whl (2.0 MB)



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import faker
import pyodbc
print("All good")

All good


In [5]:
import pyodbc
import random
from faker import Faker
from datetime import date, timedelta

fake = Faker()

conn = pyodbc.connect(
    'DRIVER={SQL Server};'
    'SERVER=GG;'
    'DATABASE=Bank;'
    'UID=sa;'
    'PWD=231931;'
)
cursor = conn.cursor()

# Branches
cities = ['Tashkent', 'Samarkand', 'Bukhara', 'Namangan', 'Andijan', 'Fergana', 'Nukus', 'Qarshi']

for i in range(1, 9):
    cursor.execute("""
        INSERT INTO Branches (branch_id, branch_name, address, city, state, country, contact_number)
        VALUES (?, ?, ?, ?, ?, ?, ?)
    """, i, f'Branch {cities[i-1]}', fake.street_address()[:30], cities[i-1], cities[i-1], 'Uzbekistan', fake.phone_number()[:30])

conn.commit()
print("Branches inserted")

Branches inserted


In [7]:
conn = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=GG;'
    'DATABASE=Bank;'
    'UID=sa;'
    'PWD=231931;'
)
cursor = conn.cursor()
print("Connected")

Connected


In [8]:
import pyodbc
print(pyodbc.drivers())

['SQL Server', 'ODBC Driver 17 for SQL Server', 'ODBC Driver 18 for SQL Server']


In [9]:
positions = ['Teller', 'Loan Officer', 'Manager', 'Analyst', 'Cashier', 'Accountant']
departments = ['Retail', 'Loans', 'Management', 'Analytics', 'Operations', 'Finance']

for i in range(1, 51):
    branch_id = random.randint(1, 8)
    cursor.execute("""
        INSERT INTO Employees (employee_id, branch_id, fullname, position, department, salary, hire_date, status)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """, i, branch_id, fake.name()[:50], random.choice(positions), random.choice(departments),
        round(random.uniform(800, 3000), 2),
        fake.date_between(start_date=date(2018, 1, 1), end_date=date(2024, 1, 1)),
        random.choice(['Active', 'Active', 'Active', 'Inactive']))

conn.commit()
print("Employees inserted")

Employees inserted


In [10]:
# Assign a manager to each branch
for i in range(1, 9):
    manager_id = random.randint(1, 50)
    cursor.execute("""
        UPDATE Branches SET manager_id = ? WHERE branch_id = ?
    """, manager_id, i)

conn.commit()
print("Managers assigned")

Managers assigned


In [11]:
# Customers - 3 segments deliberately baked in
# Segment A - High value (30%) - high income, high credit
# Segment B - Medium value (50%) - average income
# Segment C - Low value (20%) - low income, risky

employment_statuses = ['Employed', 'Self-Employed', 'Unemployed', 'Retired']

for i in range(1, 1001):
    roll = random.random()
    if roll < 0.30:
        income = round(random.uniform(5000, 20000), 2)  # High
    elif roll < 0.80:
        income = round(random.uniform(1500, 5000), 2)   # Medium
    else:
        income = round(random.uniform(200, 1500), 2)    # Low

    cursor.execute("""
        INSERT INTO Customers (customer_id, fullname, date_of_birth, email, phone_number, address, national_id, tax_id, employment_status, annual_income)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, i, fake.name()[:100],
        fake.date_of_birth(minimum_age=18, maximum_age=70),
        fake.email()[:50],
        fake.phone_number()[:50],
        fake.address()[:250],
        str(i).zfill(10),
        str(i+1000).zfill(10),
        random.choice(employment_statuses),
        income)

conn.commit()
print("Customers inserted")

Customers inserted


In [12]:
account_types = ['Savings', 'Checking', 'Business', 'Fixed Deposit']
currencies = ['UZS', 'USD', 'EUR']

for i in range(1, 1001):
    # Higher income customers get higher balances
    cursor.execute("SELECT annual_income FROM Customers WHERE customer_id = ?", i)
    income = cursor.fetchone()[0]
    
    if income > 5000:
        balance = round(random.uniform(10000, 100000), 2)
    elif income > 1500:
        balance = round(random.uniform(1000, 10000), 2)
    else:
        balance = round(random.uniform(50, 1000), 2)

    cursor.execute("""
        INSERT INTO Accounts (account_id, customer_id, branch_id, account_type, balance, currency, status, created_date)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """, i, i, random.randint(1, 8),
        random.choice(account_types),
        balance,
        random.choice(currencies),
        random.choice(['Active', 'Active', 'Active', 'Inactive', 'Suspended']),
        fake.date_between(start_date=date(2020, 1, 1), end_date=date(2024, 1, 1)))

conn.commit()
print("Accounts inserted")

Accounts inserted


In [13]:
transaction_types = ['Deposit', 'Withdrawal', 'Transfer', 'Payment']
statuses = ['Completed', 'Completed', 'Completed', 'Failed', 'Pending']

for i in range(1, 5001):
    account_id = random.randint(1, 1000)
    
    # Get account balance to make transaction amounts realistic
    cursor.execute("SELECT balance FROM Accounts WHERE account_id = ?", account_id)
    balance = cursor.fetchone()[0]
    
    # High balance accounts transact more
    if balance > 10000:
        amount = round(random.uniform(500, 10000), 2)
    elif balance > 1000:
        amount = round(random.uniform(50, 1000), 2)
    else:
        amount = round(random.uniform(5, 200), 2)

    # Fraud pattern — some transactions happen late night
    hour = random.choices([random.randint(8,18), random.randint(23,23)], weights=[90,10])[0]
    tx_date = fake.date_between(start_date=date(2024, 1, 1), end_date=date(2026, 5, 1))

    cursor.execute("""
        INSERT INTO Transactions (transaction_id, account_id, transaction_type, amount, currency, date, status, reference_no)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """, i, account_id,
        random.choice(transaction_types),
        amount,
        random.choice(currencies),
        tx_date,
        random.choice(statuses),
        fake.uuid4()[:40])

conn.commit()
print("Transactions inserted")

Transactions inserted


In [14]:
card_types = ['Visa', 'Mastercard', 'Amex']

for i in range(1, 701):  # 70% of customers have credit cards
    cursor.execute("SELECT annual_income FROM Customers WHERE customer_id = ?", i)
    income = cursor.fetchone()[0]
    
    if income > 5000:
        limit = round(random.uniform(10000, 50000), 2)
    elif income > 1500:
        limit = round(random.uniform(1000, 10000), 2)
    else:
        limit = round(random.uniform(100, 1000), 2)

    cursor.execute("""
        INSERT INTO CreditCards (card_id, customer_id, card_number, card_type, cvv, expiry_date, limit, status)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """, i, i,
        fake.credit_card_number()[:20],
        random.choice(card_types),
        random.randint(100, 999),
        fake.date_between(start_date=date(2026, 1, 1), end_date=date(2030, 1, 1)),
        limit,
        random.choice(['Active', 'Active', 'Active', 'Blocked']))

conn.commit()
print("CreditCards inserted")

# CreditCard Transactions
for i in range(1, 3001):
    card_id = random.randint(1, 700)
    cursor.execute("SELECT limit FROM CreditCards WHERE card_id = ?", card_id)
    limit = cursor.fetchone()[0]
    
    amount = round(random.uniform(10, float(limit) * 0.3), 2)

    cursor.execute("""
        INSERT INTO CreditCardTransactions (transaction_id, card_id, merchant, amount, currency, date, status)
        VALUES (?, ?, ?, ?, ?, ?, ?)
    """, i, card_id,
        fake.company()[:20],
        amount,
        random.choice(currencies),
        fake.date_between(start_date=date(2024, 1, 1), end_date=date(2026, 5, 1)),
        random.choice(['Approved', 'Approved', 'Approved', 'Declined']))

conn.commit()
print("CreditCardTransactions inserted")

CreditCards inserted
CreditCardTransactions inserted


In [15]:
loan_types = ['Personal', 'Mortgage', 'Auto', 'Business', 'Education']

for i in range(1, 401):  # 40% of customers have loans
    cursor.execute("SELECT annual_income FROM Customers WHERE customer_id = ?", i)
    income = cursor.fetchone()[0]

    if income > 5000:
        amount = round(random.uniform(20000, 200000), 2)
        interest_rate = round(random.uniform(5, 10), 2)
    elif income > 1500:
        amount = round(random.uniform(2000, 20000), 2)
        interest_rate = round(random.uniform(10, 18), 2)
    else:
        amount = round(random.uniform(500, 2000), 2)
        interest_rate = round(random.uniform(18, 30), 2)  # Risky customers pay more

    start_date = fake.date_between(start_date=date(2022, 1, 1), end_date=date(2025, 1, 1))
    end_date = start_date + timedelta(days=random.randint(365, 1825))

    cursor.execute("""
        INSERT INTO Loans (loan_id, customer_id, loan_type, amount, interest_rate, start_date, end_date, status)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """, i, i,
        random.choice(loan_types),
        amount,
        interest_rate,
        start_date,
        end_date,
        random.choice(['Active', 'Active', 'Paid', 'Defaulted']))

conn.commit()
print("Loans inserted")

# LoanPayments
payment_id = 1
for loan_id in range(1, 401):
    num_payments = random.randint(1, 12)
    cursor.execute("SELECT amount FROM Loans WHERE loan_id = ?", loan_id)
    loan_amount = cursor.fetchone()[0]
    monthly = round(float(loan_amount) / 24, 2)

    for j in range(num_payments):
        remaining = round(float(loan_amount) - (monthly * (j + 1)), 2)
        cursor.execute("""
            INSERT INTO LoanPayments (payment_id, loan_id, amount_paid, payment_date, remaining_balance)
            VALUES (?, ?, ?, ?, ?)
        """, payment_id, loan_id,
            monthly,
            fake.date_between(start_date=date(2024, 1, 1), end_date=date(2026, 5, 1)),
            max(remaining, 0))
        payment_id += 1

conn.commit()
print("LoanPayments inserted")

Loans inserted
LoanPayments inserted


In [16]:
for i in range(1, 1001):
    cursor.execute("SELECT annual_income FROM Customers WHERE customer_id = ?", i)
    income = cursor.fetchone()[0]

    if income > 5000:
        credit_score = random.randint(700, 850)
    elif income > 1500:
        credit_score = random.randint(500, 700)
    else:
        credit_score = random.randint(300, 500)  # Low income = risky

    cursor.execute("""
        INSERT INTO CreditScores (customer_id, credit_score, updated_at)
        VALUES (?, ?, ?)
    """, i, credit_score,
        fake.date_between(start_date=date(2024, 1, 1), end_date=date(2026, 1, 1)))

conn.commit()
print("CreditScores inserted")

CreditScores inserted


In [17]:
risk_levels = ['Low', 'Medium', 'High', 'Critical']

fraud_id = 1
for i in range(1, 201):  # 200 fraud cases
    cursor.execute("SELECT credit_score FROM CreditScores WHERE customer_id = ?", i)
    score = cursor.fetchone()[0]

    if score < 500:
        risk = random.choice(['High', 'High', 'Critical', 'Medium'])
    elif score < 700:
        risk = random.choice(['Medium', 'Medium', 'High', 'Low'])
    else:
        risk = random.choice(['Low', 'Low', 'Medium'])

    tx_id = random.randint(1, 5000)

    cursor.execute("""
        INSERT INTO FraudDetection (fraud_id, customer_id, transaction_id, risk_level, reported_date)
        VALUES (?, ?, ?, ?, ?)
    """, fraud_id, i, tx_id, risk,
        fake.date_between(start_date=date(2024, 1, 1), end_date=date(2026, 5, 1)))
    
    fraud_id += 1

conn.commit()
print("FraudDetection inserted")

FraudDetection inserted


In [18]:
# Investments - high income customers invest more
investment_types = ['Stocks', 'Bonds', 'Real Estate', 'Mutual Funds', 'Crypto']

for i in range(1, 401):
    cursor.execute("SELECT annual_income FROM Customers WHERE customer_id = ?", i)
    income = cursor.fetchone()[0]

    if income > 5000:
        amount = round(random.uniform(5000, 50000), 2)
        roi = round(random.uniform(8, 20), 2)
    elif income > 1500:
        amount = round(random.uniform(500, 5000), 2)
        roi = round(random.uniform(3, 8), 2)
    else:
        amount = round(random.uniform(50, 500), 2)
        roi = round(random.uniform(1, 3), 2)

    cursor.execute("""
        INSERT INTO Investments (investment_id, customer_id, investment_type, amount, ROI, MaturityDate)
        VALUES (?, ?, ?, ?, ?, ?)
    """, i, i,
        random.choice(investment_types),
        amount,
        roi,
        fake.date_between(start_date=date(2026, 1, 1), end_date=date(2030, 1, 1)))

conn.commit()
print("Investments inserted")

# DebtCollection - low income customers mostly
for i in range(1, 151):
    cursor.execute("""
        INSERT INTO DebtCollection (dept_id, customer_id, amount_due, due_date, collector_assigned)
        VALUES (?, ?, ?, ?, ?)
    """, i, random.randint(800, 1000),
        round(random.uniform(100, 5000), 2),
        fake.date_between(start_date=date(2024, 1, 1), end_date=date(2026, 12, 1)),
        fake.name()[:30])

conn.commit()
print("DebtCollection inserted")

# BillPayments
for i in range(1, 2001):
    cursor.execute("""
        INSERT INTO BillPayments (payment_id, customer_id, biller_name, amount, date, status)
        VALUES (?, ?, ?, ?, ?, ?)
    """, i, random.randint(1, 1000),
        random.choice(['Electricity', 'Internet', 'Gas', 'Water', 'Phone'])[:30],
        round(random.uniform(10, 500), 2),
        fake.date_between(start_date=date(2024, 1, 1), end_date=date(2026, 5, 1)),
        random.choice(['Paid', 'Paid', 'Paid', 'Pending', 'Failed']))

conn.commit()
print("BillPayments inserted")

Investments inserted
DebtCollection inserted
BillPayments inserted
